# Short-seller reports and company responses

Find companies that responded to a short-seller report in an 8-K, and those that list a past report as a risk factor.

Read it with charts at https://www.quantisnow.com/sec-api/themes/short-seller-reports. More notebooks: https://github.com/quantisnow/sec-edgar-notebooks. Set `QUANTISNOW_API_KEY` before running; make a key at https://www.quantisnow.com/sec-api/keys.

Short sellers publish outside the SEC, but targets often answer in an 8-K, and later list the report and any lawsuits in their risk factors.

## Companies whose filings mention short-seller reports

### Which filings respond to or mention a short-seller report?

8-Ks with "refutes" or "categorically" near the phrase are the direct responses.

```shell
curl -X POST https://www.quantisnow.com/api/v1/full-text \
  -H "Authorization: Bearer $QUANTISNOW_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{
  "query": "\"short seller\" OR \"short-seller\" OR \"short report\"",
  "formTypes": ["8-K", "10-Q", "10-K", "6-K"],
  "sort": "filedAt:desc",
  "size": 5,
  "facets": true
}'
```

Response (real, first 2 of 26):

```json
{
  "total": {
    "value": 26,
    "relation": "eq"
  },
  "took": 123,
  "data": [
    {
      "id": "0001868941-26-000057",
      "kind": "filing",
      "source": "SEC",
      "accessionNo": "0001868941-26-000057",
      "formType": "10-K",
      "filedAt": "2026-11-23T16:12:45-05:00",
      "cik": "0001868941",
      "companyName": "Fluence Energy Inc.",
      "ticker": "FLNC",
      "tickers": [
        "FLNC"
      ],
      "items": [],
      "description": "SEC Form 10-K filed by Fluence Energy Inc.",
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/1868941/000186894126000057/0001868941-26-000057-index.htm",
      "quantisnowUrl": "https://www.quantisnow.com/insight/sec-form-10-k-filed-by-fluence-energy-inc-6804007",
      "score": null,
      "highlights": [
        "Accordingly, it is in the interest of a <em>short seller</em> of our Class A common stock for the price to decline.",
        "For example, on February 22, 2024, as previously disclosed, a <em>short seller</em> report was published which contained certain allegations against the Company (the \"<em>Short Seller</em> Report\").",
        "These areas were the subject of assertions in the <em>Short Seller</em> Report and were included in the Audit Committee's internal investigation."
      ]
    },
    {
      "id": "0001868941-26-000056",
      "kind": "filing",
      "source": "SEC",
      "accessionNo": "0001868941-26-000056",
      "formType": "8-K",
      "filedAt": "2026-11-23T16:11:34-05:00",
      "cik": "0001868941",
      "companyName": "Fluence Energy Inc.",
      "ticker": "FLNC",
      "tickers": [
        "FLNC"
      ],
      "items": [
        "2.02",
        "9.01"
      ],
      "description": "Fluence Energy Inc. filed SEC Form 8-K: Results of Operations and Financial Condition, Financial Statements and Exhibits",
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/1868941/000186894126000056/0001868941-26-000056-index.htm",
      "quantisnowUrl": "https://www.quantisnow.com/insight/fluence-energy-inc-filed-sec-form-8-k-results-of-operations-6803998",
      "score": null,
      "highlights": [
        "stock for the foreseeable future; being a \"controlled company\" within the meaning of the Nasdaq rules; conflicts of interest by officers and directors due to positions with our continuing equity owners; future dilution; <em>short</em>-<em>seller</em>"
      ]
    }
  ],
  "facets": {
    "formType": [
      {
        "value": "10-Q",
        "count": 15
      },
      {
        "value": "8-K",
        "count": 6
      },
      {
        "value": "10-K",
        "count": 3
      },
      {
        "value": "6-K",
        "count": 2
      }
    ],
    "month": [
      {
        "value": "2026-11",
        "count": 23
      },
      {
        "value": "2026-10",
        "count": 3
      }
    ],
    "companyName": [
      {
        "value": "Nutex Health Inc.",
        "count": 3
      },
      {
        "value": "Fluence Energy Inc.",
        "count": 2
      },
      {
        "value": "Applovin Corporation",
        "count": 1
      },
      {
        "value": "Block Inc.",
        "count": 1
      },
      {
        "value": "CareDx Inc.",
        "count": 1
      },
      {
        "value": "Corcept Therapeutics Incorporated",
        "count": 1
      },
      {
        "value": "Deep Fission Inc.",
        "count": 1
      },
      {
        "value": "Enovix Corporation",
        "count": 1
      },
      {
        "value": "Equinix Inc.",
        "count": 1
      },
      {
        "value": "Evolution Metals & Technologies Corp.",
        "count": 1
      }
    ],
    "source": [
      {
        "value": "SEC",
        "count": 26
      }
    ]
  }
}
```

## A screener for this theme

Two calls: full-text search for the companies, then open-market insider buys in them. Change the query or the date range and run it again.

In [ ]:
import os, requests

API = "https://www.quantisnow.com/api/v1"
H = {"Authorization": f"Bearer {os.environ['QUANTISNOW_API_KEY']}"}

# 1. Companies whose filings talk about the theme
hits = requests.post(f"{API}/full-text", headers=H, json={
    "query": "\"short seller\" OR \"short-seller\" OR \"short report\"",
    "formTypes": ["8-K","10-Q","10-K","6-K"],
    "sort": "filedAt:desc", "size": 50,
}).json()["data"]
tickers = sorted({h["ticker"] for h in hits if h["ticker"]})
print(len(tickers), "companies:", ", ".join(tickers))

# 2. Open-market insider buys in those companies, last 90 days
if tickers:
    buys = requests.post(f"{API}/insider-trades", headers=H, json={
        "query": f"issuer.ticker:({' OR '.join(tickers[:40])}) AND transactionCodes:P AND filedAt:[2026-08-26 TO 2026-11-24]",
        "sort": "purchased.valueUsd:desc", "size": 50,
    }).json()["data"]
    for b in buys:
        print(b["issuer"]["ticker"], b["reportingOwner"]["name"], round(b["purchased"]["valueUsd"]))

## From the command line

```shell
npx quantisnow search '"short seller" OR "short-seller" OR "short report"' --forms 8-K,10-Q,10-K,6-K --newest
```

## Ask Claude or ChatGPT

With the [MCP server](https://www.quantisnow.com/sec-api/mcp) connected, paste this:

```text
Using quantisnow, find 8-Ks from the last 90 days where a company responds to a short-seller report, and check each for an Item 4.01 or NT 10-Q since.
```

## Reading the results

Watch what follows: an auditor change, a delayed 10-Q or a special committee review in the next filings says more than the rebuttal.

> These pages describe what filings show. Nothing here is a recommendation to buy or sell any security.

Related: [Auditor resignations](https://www.quantisnow.com/sec-api/guides/auditor-resignations-8-k-4-01), [Late filers](https://www.quantisnow.com/sec-api/guides/late-filings-nt-10-k-nt-10-q).